# Define Address Mapping

Notebook này dùng để tạo file mapping giữa **mã file Dxxx** và thông tin địa lý ở mức:

- `district_city`: quận/huyện/thị xã/thành phố
- `province`: tỉnh/thành phố cấp tỉnh

Notebook hỗ trợ 2 cách chạy:

1. Đọc trực tiếp từ các file `RAW/foody_raw/D` và `RAW/shopee_raw/D` nếu có dữ liệu gốc.
2. Nếu không có dữ liệu gốc, đọc từ file `area_code_mapping_template.csv` đã tạo trước đó.

Output cuối cùng là file:

```text
OUTPUT/area_location_mapping.csv
```


## 1. Import & Config


In [1]:
# Import các thư viện cần thiết

import re
import pandas as pd
import numpy as np

from pathlib import Path

# Cấu hình hiển thị dataframe cho dễ xem
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 300)


In [2]:
# Kiểm tra thư mục hiện tại notebook đang chạy
print("Thư mục hiện tại:", Path.cwd())

Thư mục hiện tại: d:\HOC_TREN_TRUONG\TIEN_XU_LY&XD_DL\DO_AN\notebook\DS108.Q21\notebooks\EDA


In [3]:
# Khai báo thư mục project
DATA_FOLDER = (Path.cwd().resolve().parent.parent / 'data')
RAW_DIR = DATA_FOLDER / "data_raw"
FOODY_RAW_DIR = RAW_DIR / "foody_csv"
SHOPEE_RAW_DIR = RAW_DIR / "shopee_csv"
FOODY_D_DIR = FOODY_RAW_DIR / "D"
SHOPEE_D_DIR = SHOPEE_RAW_DIR / "D"

OUTPUT_DIR = Path.cwd()

# File template dùng làm phương án dự phòng nếu không đọc trực tiếp từ RAW được
TEMPLATE_PATH = Path.cwd() / "area_code_mapping_template.csv"

print("DATA_FOLDER:", DATA_FOLDER)
print("FOODY_D_DIR tồn tại:", FOODY_D_DIR.exists())
print("SHOPEE_D_DIR tồn tại:", SHOPEE_D_DIR.exists())
print("TEMPLATE_PATH tồn tại:", TEMPLATE_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


DATA_FOLDER: D:\HOC_TREN_TRUONG\TIEN_XU_LY&XD_DL\DO_AN\notebook\DS108.Q21\data
FOODY_D_DIR tồn tại: True
SHOPEE_D_DIR tồn tại: True
TEMPLATE_PATH tồn tại: False
OUTPUT_DIR: d:\HOC_TREN_TRUONG\TIEN_XU_LY&XD_DL\DO_AN\notebook\DS108.Q21\notebooks\EDA


## 2. Hàm đọc dữ liệu


In [4]:
def get_data_files(folder):
    """
    Lấy danh sách file dữ liệu trong một thư mục.
    Hỗ trợ các định dạng csv, xlsx, xls.
    """

    if not folder.exists():
        return []

    files = (
        list(folder.glob("*.csv")) +
        list(folder.glob("*.xlsx")) +
        list(folder.glob("*.xls"))
    )

    return sorted(files)


def read_data_file(file_path):
    """
    Đọc file dữ liệu theo phần mở rộng.
    """

    file_extension = file_path.suffix.lower()

    if file_extension == ".csv":
        return pd.read_csv(file_path)

    if file_extension in [".xlsx", ".xls"]:
        return pd.read_excel(file_path)

    raise ValueError(f"Định dạng file chưa được hỗ trợ: {file_extension}")


def extract_area_code(file_path_or_name):
    """
    Lấy mã khu vực từ tên file.

    Ví dụ:
    - D108.csv -> 108
    - C108.csv -> 108
    - 108.csv -> 108
    """

    file_name = Path(str(file_path_or_name)).stem
    file_name = file_name.strip()

    # Bỏ ký tự C hoặc D ở đầu tên file nếu có
    if len(file_name) > 0 and file_name[0].upper() in ["C", "D"]:
        return file_name[1:]

    return file_name


## 3. Đọc dữ liệu địa chỉ

Notebook ưu tiên đọc trực tiếp từ file D gốc để lấy cột `Address`. Nếu không tìm thấy file D, notebook sẽ đọc từ `area_code_mapping_template.csv` và dùng cột `address_examples`.


In [5]:
# Khai báo nguồn dữ liệu file D

DATA_SOURCES = {
    "foody": {
        "restaurant_dir": FOODY_D_DIR
    },
    "shopee": {
        "restaurant_dir": SHOPEE_D_DIR
    }
}

# Kiểm tra số file D ở từng nguồn

for source_name, paths in DATA_SOURCES.items():
    restaurant_files = get_data_files(paths["restaurant_dir"])
    print(f"{source_name}: {len(restaurant_files)} file D")


foody: 118 file D
shopee: 88 file D


In [6]:
# Đọc trực tiếp các file D nếu có
# Kết quả tạo ra raw_address_data với ít nhất các cột:
# source, area_code, source_file, Address

restaurant_dfs = []
restaurant_file_errors = []

for source_name, paths in DATA_SOURCES.items():
    restaurant_files = get_data_files(paths["restaurant_dir"])

    for file in restaurant_files:
        try:
            df = read_data_file(file)

            # Chỉ xử lý nếu file có cột Address
            if "Address" not in df.columns:
                restaurant_file_errors.append({
                    "source": source_name,
                    "file_name": file.name,
                    "error": "Không tìm thấy cột Address"
                })
                continue

            df = df.copy()
            df["source"] = source_name
            df["area_code"] = extract_area_code(file.name)
            df["source_file"] = file.name

            restaurant_dfs.append(df[["source", "area_code", "source_file", "Address"]])

        except Exception as e:
            restaurant_file_errors.append({
                "source": source_name,
                "file_name": file.name,
                "error": str(e)
            })

if len(restaurant_dfs) > 0:
    raw_address_data = pd.concat(restaurant_dfs, ignore_index=True)
    data_mode = "raw_d_files"
else:
    raw_address_data = pd.DataFrame()
    data_mode = None

print("Chế độ đọc dữ liệu:", data_mode)
print("Kích thước raw_address_data:", raw_address_data.shape)
print("Số file lỗi:", len(restaurant_file_errors))

pd.DataFrame(restaurant_file_errors).head()


Chế độ đọc dữ liệu: raw_d_files
Kích thước raw_address_data: (19028, 4)
Số file lỗi: 0


""


In [7]:
# Nếu không đọc được file D gốc, dùng file area_code_mapping_template.csv
# File template thường có cột address_examples chứa nhiều địa chỉ mẫu được nối bằng dấu |

if raw_address_data.empty:
    if not TEMPLATE_PATH.exists():
        raise FileNotFoundError(
            "Không tìm thấy file D gốc và cũng không tìm thấy area_code_mapping_template.csv. "
            "Hãy kiểm tra lại PROJECT_ROOT hoặc đặt file template vào thư mục project."
        )

    template_df = pd.read_csv(TEMPLATE_PATH)

    required_cols = ["source", "area_code", "source_file", "address_examples"]
    missing_cols = [col for col in required_cols if col not in template_df.columns]

    if len(missing_cols) > 0:
        raise ValueError(f"File template thiếu các cột: {missing_cols}")

    rows = []

    for _, row in template_df.iterrows():
        address_examples = row["address_examples"]

        if pd.isna(address_examples):
            addresses = []
        else:
            addresses = [
                address.strip()
                for address in str(address_examples).split("|")
                if address.strip()
            ]

        for address in addresses:
            rows.append({
                "source": row["source"],
                "area_code": str(row["area_code"]),
                "source_file": row["source_file"],
                "Address": address
            })

    raw_address_data = pd.DataFrame(rows)
    data_mode = "template_file"

print("Chế độ đọc dữ liệu cuối cùng:", data_mode)
print("Kích thước raw_address_data:", raw_address_data.shape)
raw_address_data.head(10)


Chế độ đọc dữ liệu cuối cùng: raw_d_files
Kích thước raw_address_data: (19028, 4)


,source,area_code,source_file,Address
0,foody,10,D10.csv,"249 Thích Quảng Đức, P. Chánh Nghĩa, Thành Phố Thủ Dầu Một, Bình Dương"
1,foody,10,D10.csv,"B70B Cách Mạng Tháng 8, P. Bình Nhâm, Thị Xã Thuận An, Bình Dương"
2,foody,10,D10.csv,"Tầng Trệt Toà Nhà Sora Gardens II, Lô C17 Đại Lộ Hùng Vương, P. Phú Mỹ, Thành Phố Thủ Dầu Một, Bình Dương"
3,foody,10,D10.csv,"Ô 47 - DC29 Đường D1, KDC Việt Sing, P. An Phú, Thị Xã Thuận An, Bình Dương"
4,foody,10,D10.csv,"21 Phạm Ngũ Lão, P. Phú Cường, Thành Phố Thủ Dầu Một, Bình Dương"
5,foody,10,D10.csv,"168 Trần Văn Ơn, P. Phú Lợi, Thành Phố Thủ Dầu Một, Bình Dương"
6,foody,10,D10.csv,"6 Nguyễn An Ninh, P. Dĩ An, Thị Xã Dĩ An, Bình Dương"
7,foody,10,D10.csv,"19 Nguyễn Du, P. Dĩ An, Thị Xã Dĩ An, Bình Dương"
8,foody,10,D10.csv,"314 Đồng Khởi, Khu Phố 3, P. Hòa Phú, Thành Phố Thủ Dầu Một, Bình Dương"
9,foody,10,D10.csv,"39/2/78 Đ. Thống Nhất, KP Nội Hoá 1, P. Bình An, Thị Xã Dĩ An, Bình Dương"


## 4. Hàm tách quận/huyện/thành phố và tỉnh

Nguyên tắc xử lý:

- Tách địa chỉ theo dấu phẩy.
- Bỏ qua cấp phường/xã/thị trấn.
- Ưu tiên nhận diện cấp quận/huyện/thị xã/thành phố.
- Tỉnh/thành phố cấp tỉnh thường là thành phần cuối cùng của địa chỉ.


In [8]:
def normalize_text(text):
    """
    Chuẩn hóa chuỗi địa chỉ ở mức cơ bản.
    """

    if pd.isna(text):
        return None

    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    if text == "" or text.lower() in ["nan", "none"]:
        return None

    return text


def split_address_parts(address):
    """
    Tách địa chỉ thành các phần theo dấu phẩy.
    """

    if pd.isna(address):
        return []

    parts = [normalize_text(part) for part in str(address).split(",")]
    parts = [part for part in parts if part is not None]

    return parts


def is_ward_level(text):
    """
    Kiểm tra một phần địa chỉ có phải cấp phường/xã/thị trấn không.
    """

    text = normalize_text(text)

    if text is None:
        return False

    ward_patterns = [
        r"^P\.?\s*\d+",
        r"^P\.",
        r"^Phường\b",
        r"^Xã\b",
        r"^Thị trấn\b",
        r"^TT\."
    ]

    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in ward_patterns)


def is_district_level(text):
    """
    Kiểm tra một phần địa chỉ có phải cấp quận/huyện/thị xã/thành phố không.
    """

    text = normalize_text(text)

    if text is None:
        return False

    if is_ward_level(text):
        return False

    district_patterns = [
        r"^Quận\b",
        r"^Q\.",
        r"^Huyện\b",
        r"^H\.",
        r"^Thị xã\b",
        r"^Thị Xã\b",
        r"^TX\.",
        r"^Thành phố\b",
        r"^Thành Phố\b",
        r"^TP\.",
        r"^Tp\.",
        r"^TP\b",
        r"^Tp\b"
    ]

    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in district_patterns)


def standardize_province_name(province):
    """
    Chuẩn hóa nhẹ tên tỉnh/thành phố để output dễ đọc hơn.
    Không ép quá mạnh để tránh làm sai tên địa phương.
    """

    province = normalize_text(province)

    if province is None:
        return None

    replace_map = {
        "TP HCM": "TP. HCM",
        "TP.HCM": "TP. HCM",
        "Tp HCM": "TP. HCM",
        "Tp. HCM": "TP. HCM",
        "HCM": "TP. HCM",
        "Hồ Chí Minh": "TP. HCM",
        "TP Hồ Chí Minh": "TP. HCM",
        "TP. Hồ Chí Minh": "TP. HCM",
        "Tp. Hồ Chí Minh": "TP. HCM"
    }

    return replace_map.get(province, province)


def extract_district_and_province_from_one_address(address):
    """
    Tách district_city và province từ một địa chỉ đơn lẻ.

    Ví dụ:
    'P. 1, Quận 5, TP. HCM'
    -> district_city = 'Quận 5'
    -> province = 'TP. HCM'
    """

    parts = split_address_parts(address)

    if len(parts) == 0:
        return pd.Series([None, None])

    # Tỉnh/thành phố cấp tỉnh thường nằm ở phần cuối cùng
    province = standardize_province_name(parts[-1])

    district_city = None

    # Tìm từ cuối lên đầu, bỏ qua phần tỉnh/thành phố cấp tỉnh
    # Ưu tiên phần có từ khóa quận/huyện/thị xã/thành phố
    for part in reversed(parts[:-1]):
        if is_district_level(part):
            district_city = part
            break

    # Nếu không tìm được, lấy phần gần cuối nhất nhưng không phải phường/xã
    if district_city is None:
        for part in reversed(parts[:-1]):
            if not is_ward_level(part):
                district_city = part
                break

    return pd.Series([district_city, province])


## 5. Áp dụng tách địa chỉ


In [9]:
# Tách district_city và province từ cột Address

raw_address_data[["district_city", "province"]] = (
    raw_address_data["Address"]
    .apply(extract_district_and_province_from_one_address)
)

raw_address_data[[
    "source",
    "area_code",
    "source_file",
    "Address",
    "district_city",
    "province"
]].head(10)


,source,area_code,source_file,Address,district_city,province
0,foody,10,D10.csv,"249 Thích Quảng Đức, P. Chánh Nghĩa, Thành Phố Thủ Dầu Một, Bình Dương",Thành Phố Thủ Dầu Một,Bình Dương
1,foody,10,D10.csv,"B70B Cách Mạng Tháng 8, P. Bình Nhâm, Thị Xã Thuận An, Bình Dương",Thị Xã Thuận An,Bình Dương
2,foody,10,D10.csv,"Tầng Trệt Toà Nhà Sora Gardens II, Lô C17 Đại Lộ Hùng Vương, P. Phú Mỹ, Thành Phố Thủ Dầu Một, Bình Dương",Thành Phố Thủ Dầu Một,Bình Dương
3,foody,10,D10.csv,"Ô 47 - DC29 Đường D1, KDC Việt Sing, P. An Phú, Thị Xã Thuận An, Bình Dương",Thị Xã Thuận An,Bình Dương
4,foody,10,D10.csv,"21 Phạm Ngũ Lão, P. Phú Cường, Thành Phố Thủ Dầu Một, Bình Dương",Thành Phố Thủ Dầu Một,Bình Dương
5,foody,10,D10.csv,"168 Trần Văn Ơn, P. Phú Lợi, Thành Phố Thủ Dầu Một, Bình Dương",Thành Phố Thủ Dầu Một,Bình Dương
6,foody,10,D10.csv,"6 Nguyễn An Ninh, P. Dĩ An, Thị Xã Dĩ An, Bình Dương",Thị Xã Dĩ An,Bình Dương
7,foody,10,D10.csv,"19 Nguyễn Du, P. Dĩ An, Thị Xã Dĩ An, Bình Dương",Thị Xã Dĩ An,Bình Dương
8,foody,10,D10.csv,"314 Đồng Khởi, Khu Phố 3, P. Hòa Phú, Thành Phố Thủ Dầu Một, Bình Dương",Thành Phố Thủ Dầu Một,Bình Dương
9,foody,10,D10.csv,"39/2/78 Đ. Thống Nhất, KP Nội Hoá 1, P. Bình An, Thị Xã Dĩ An, Bình Dương",Thị Xã Dĩ An,Bình Dương


In [10]:
# Kiểm tra các dòng chưa tách được district_city hoặc province

unparsed_rows = raw_address_data[
    raw_address_data["district_city"].isna() |
    raw_address_data["province"].isna()
]

print("Số dòng chưa tách được đầy đủ:", len(unparsed_rows))
unparsed_rows.head(10)


Số dòng chưa tách được đầy đủ: 47


,source,area_code,source_file,Address,district_city,province
167,foody,100,D100.csv,NaN,NaN,NaN
1725,foody,104,D104.csv,NaN,NaN,NaN
1921,foody,104,D104.csv,NaN,NaN,NaN
2258,foody,104,D104.csv,NaN,NaN,NaN
2260,foody,104,D104.csv,NaN,NaN,NaN
2273,foody,104,D104.csv,NaN,NaN,NaN
2572,foody,105,D105.csv,NaN,NaN,NaN
3219,foody,106,D106.csv,NaN,NaN,NaN
3257,foody,106,D106.csv,NaN,NaN,NaN
3367,foody,106,D106.csv,NaN,NaN,NaN


## 6. Gom kết quả theo từng file

Mỗi file Dxxx thường tương ứng với một khu vực. Vì vậy, sau khi tách địa chỉ cho từng dòng, notebook lấy giá trị xuất hiện nhiều nhất trong từng file.


In [11]:
def get_most_common_value(series):
    """
    Lấy giá trị xuất hiện nhiều nhất trong một Series.
    Nếu toàn bộ bị thiếu thì trả về None.
    """

    values = series.dropna().astype(str).str.strip()
    values = values[values != ""]

    if len(values) == 0:
        return None

    return values.value_counts().idxmax()


def get_top_values_text(series, top_n=3):
    """
    Lấy top giá trị phổ biến nhất để hỗ trợ kiểm tra thủ công.
    """

    values = series.dropna().astype(str).str.strip()
    values = values[values != ""]

    if len(values) == 0:
        return ""

    counts = values.value_counts().head(top_n)

    return " | ".join([f"{idx} ({count})" for idx, count in counts.items()])


In [12]:
# Tạo bảng mapping theo từng file
# district_city và province được lấy theo giá trị phổ biến nhất trong file

area_location_mapping = (
    raw_address_data
    .groupby(["source", "area_code", "source_file"])
    .agg(
        district_city=("district_city", get_most_common_value),
        province=("province", get_most_common_value),
        num_addresses=("Address", "count"),
        top_district_candidates=("district_city", get_top_values_text),
        top_province_candidates=("province", get_top_values_text)
    )
    .reset_index()
)

area_location_mapping.head(10)


,source,area_code,source_file,district_city,province,num_addresses,top_district_candidates,top_province_candidates
0,foody,10,D10.csv,Thành Phố Thủ Dầu Một,Bình Dương,76,Thành Phố Thủ Dầu Một (31) | Thị Xã Thuận An (19) | Thị Xã Dĩ An (19),Bình Dương (76)
1,foody,100,D100.csv,Quận 5,TP. HCM,115,Quận 5 (78) | Quận 1 (8) | Quận Tân Bình (5),TP. HCM (115)
2,foody,1000,D1000.csv,Thành Phố Long Xuyên,An Giang,35,Thành Phố Long Xuyên (27) | Thành Phố Châu Đốc (5) | Huyện Châu Phú (1),An Giang (35)
3,foody,101,D101.csv,Quận Tân Phú,TP. HCM,700,Quận Tân Phú (401) | Quận 1 (45) | Quận 7 (34),TP. HCM (700)
4,foody,102,D102.csv,Quận 5,TP. HCM,123,Quận 5 (55) | Quận 1 (19) | Quận Tân Bình (10),TP. HCM (123)
5,foody,103,D103.csv,Quận Gò Vấp,TP. HCM,660,Quận Gò Vấp (342) | Quận 1 (44) | Tp. Thủ Đức (37),TP. HCM (660)
6,foody,104,D104.csv,Quận Bình Thạnh,TP. HCM,783,Quận Bình Thạnh (408) | Quận 1 (63) | Tp. Thủ Đức (49),TP. HCM (783)
7,foody,105,D105.csv,Quận 12,TP. HCM,421,Quận 12 (270) | Tp. Thủ Đức (21) | Quận 7 (20),TP. HCM (421)
8,foody,106,D106.csv,Quận Tân Bình,TP. HCM,643,Quận Tân Bình (339) | Quận 1 (48) | Quận Bình Thạnh (33),TP. HCM (643)
9,foody,107,D107.csv,Quận Bình Tân,TP. HCM,525,Quận Bình Tân (308) | Quận 7 (26) | Quận 1 (26),TP. HCM (525)


## 7. Kiểm tra các file nghi ngờ

Các file cần kiểm tra lại gồm:

- Không tách được quận/huyện/thành phố hoặc tỉnh.
- Có nhiều ứng viên tỉnh/thành khác nhau trong cùng một file.


In [13]:
# File chưa tách được đầy đủ district_city hoặc province

missing_mapping = area_location_mapping[
    area_location_mapping["district_city"].isna() |
    area_location_mapping["province"].isna()
]

print("Số file chưa tách được đầy đủ:", len(missing_mapping))
missing_mapping


Số file chưa tách được đầy đủ: 0


,source,area_code,source_file,district_city,province,num_addresses,top_district_candidates,top_province_candidates


In [14]:
# Kiểm tra nhanh một file cụ thể nếu thấy nghi ngờ
# Đổi tên file ở biến file_check để xem địa chỉ gốc và kết quả tách

file_check = area_location_mapping["source_file"].iloc[0]

raw_address_data[
    raw_address_data["source_file"] == file_check
][[
    "Address",
    "district_city",
    "province"
]].head(10)


,Address,district_city,province
0,"249 Thích Quảng Đức, P. Chánh Nghĩa, Thành Phố Thủ Dầu Một, Bình Dương",Thành Phố Thủ Dầu Một,Bình Dương
1,"B70B Cách Mạng Tháng 8, P. Bình Nhâm, Thị Xã Thuận An, Bình Dương",Thị Xã Thuận An,Bình Dương
2,"Tầng Trệt Toà Nhà Sora Gardens II, Lô C17 Đại Lộ Hùng Vương, P. Phú Mỹ, Thành Phố Thủ Dầu Một, Bình Dương",Thành Phố Thủ Dầu Một,Bình Dương
3,"Ô 47 - DC29 Đường D1, KDC Việt Sing, P. An Phú, Thị Xã Thuận An, Bình Dương",Thị Xã Thuận An,Bình Dương
4,"21 Phạm Ngũ Lão, P. Phú Cường, Thành Phố Thủ Dầu Một, Bình Dương",Thành Phố Thủ Dầu Một,Bình Dương
5,"168 Trần Văn Ơn, P. Phú Lợi, Thành Phố Thủ Dầu Một, Bình Dương",Thành Phố Thủ Dầu Một,Bình Dương
6,"6 Nguyễn An Ninh, P. Dĩ An, Thị Xã Dĩ An, Bình Dương",Thị Xã Dĩ An,Bình Dương
7,"19 Nguyễn Du, P. Dĩ An, Thị Xã Dĩ An, Bình Dương",Thị Xã Dĩ An,Bình Dương
8,"314 Đồng Khởi, Khu Phố 3, P. Hòa Phú, Thành Phố Thủ Dầu Một, Bình Dương",Thành Phố Thủ Dầu Một,Bình Dương
9,"39/2/78 Đ. Thống Nhất, KP Nội Hoá 1, P. Bình An, Thị Xã Dĩ An, Bình Dương",Thị Xã Dĩ An,Bình Dương


## 8. Xuất file mapping

File chính nên dùng để merge lại vào notebook EDA là:

```text
OUTPUT/area_location_mapping.csv
```

File này giữ thêm `source` và `area_code` để tránh nhầm giữa Foody và Shopee nếu hai nguồn có cùng mã file.


In [15]:
# Tạo file mapping chính để dùng về sau

final_mapping = area_location_mapping[[
    "source",
    "area_code",
    "source_file",
    "district_city",
    "province"
]].copy()

final_mapping_path = OUTPUT_DIR / "area_location_mapping.csv"

final_mapping.to_csv(
    final_mapping_path,
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file:", final_mapping_path)
final_mapping.head(10)


Đã xuất file: d:\HOC_TREN_TRUONG\TIEN_XU_LY&XD_DL\DO_AN\notebook\DS108.Q21\notebooks\EDA\area_location_mapping.csv


,source,area_code,source_file,district_city,province
0,foody,10,D10.csv,Thành Phố Thủ Dầu Một,Bình Dương
1,foody,100,D100.csv,Quận 5,TP. HCM
2,foody,1000,D1000.csv,Thành Phố Long Xuyên,An Giang
3,foody,101,D101.csv,Quận Tân Phú,TP. HCM
4,foody,102,D102.csv,Quận 5,TP. HCM
5,foody,103,D103.csv,Quận Gò Vấp,TP. HCM
6,foody,104,D104.csv,Quận Bình Thạnh,TP. HCM
7,foody,105,D105.csv,Quận 12,TP. HCM
8,foody,106,D106.csv,Quận Tân Bình,TP. HCM
9,foody,107,D107.csv,Quận Bình Tân,TP. HCM


In [16]:
# Tạo thêm file kiểm tra chi tiết nếu cần debug
# File này có thêm top candidate để xem vì sao notebook chọn district/province như vậy

debug_mapping_path = OUTPUT_DIR / "area_location_mapping_debug.csv"

area_location_mapping.to_csv(
    debug_mapping_path,
    index=False,
    encoding="utf-8-sig"
)

print("Đã xuất file debug:", debug_mapping_path)


Đã xuất file debug: d:\HOC_TREN_TRUONG\TIEN_XU_LY&XD_DL\DO_AN\notebook\DS108.Q21\notebooks\EDA\area_location_mapping_debug.csv
